In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import distinctipy
import matplotlib.colors as mcolors
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import mannwhitneyu
from plottable import ColumnDefinition, Table
from plottable.plots import bar
import ast
import copy
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/age-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx", index_col=0)
imms = df_feats.index.to_list()
df = pd.read_excel(f"{path_data}/data.xlsx")

statuses_rename = {x: x.replace(' ', '\n').replace('-', '\n') for x in df['Status'].value_counts().index.values}
df['Status Origin'] = df['Status']
df['Status'] = df['Status'].replace(statuses_rename)
status_count = df['Status'].value_counts()
statuses = status_count.index.values

df_groups = pd.read_excel(f"{path_data}/groups.xlsx", index_col=0)
icd_chpts = np.sort(df_groups['ICD-11 chapter'].unique())
icd_cols = []
for icd_chpt in icd_chpts:
    icd_cols.append(f'Passed\nICD-11\nChapter {icd_chpt}')
icd_cols_max = [f"Max\n{x}" for x in icd_cols]
icd_chpts_names_dict = pd.Series(df_groups['ICD-11 name'].values, index=df_groups['ICD-11 chapter']).to_dict()

# Prepare colors
def make_rgb_transparent(rgb, bg_rgb, alpha):
    return [alpha * c1 + (1 - alpha) * c2 for (c1, c2) in zip(rgb, bg_rgb)]

def form_bar(base):
    def formatter(x):
        return f'{str(int(round(x * base)))}/{base}'
    return formatter

# Colors for ICD-11 chapters
colors = distinctipy.get_colors(len(icd_chpts), [mcolors.hex2color(mcolors.CSS4_COLORS['black']), mcolors.hex2color(mcolors.CSS4_COLORS['white'])], rng=1337, pastel_factor=0.5)
colors_icd_chpts = {icd_chpt: colors[icd_chpt_id] for icd_chpt_id, icd_chpt in enumerate(icd_chpts)}
colormaps_icd_chpts = {
    icd_chpt: LinearSegmentedColormap.from_list(
        name=f"ICD-11 Chapter {icd_chpt} cmap",
        colors=[make_rgb_transparent(colors_icd_chpts[icd_chpt], (1, 1, 1), 0.2), colors_icd_chpts[icd_chpt]], N=256
    )
    for icd_chpt in icd_chpts
}
colormap_total = LinearSegmentedColormap.from_list(
    name=f"ICD-11 Total cmap",
    colors=[
        mcolors.hex2color(mcolors.CSS4_COLORS['lavender']),
        mcolors.hex2color(mcolors.CSS4_COLORS['dimgray'])],
    N=256,
)

# Prepare data for the figure
imms_tests = pd.DataFrame(index=imms)
new_cols = ['Passed\nICD-11\nTotal'] + icd_cols + ['Max\nPassed\nICD-11\nTotal'] + icd_cols_max
for col in new_cols: 
    imms_tests[col] = None
    
for imm in (pbar := tqdm(imms)):
    pbar.set_description(f"Processing {imm}")
        
    for section_id, section_row in df_groups.iterrows():
        section_statuses = ast.literal_eval(section_row['Statuses'])
        section_groups = ast.literal_eval(section_row['Groups'])
        section_directions = ast.literal_eval(section_row['Directions'])
        df_section = df.loc[(df['GSE'] == section_row['GSE']) & (df['Status Origin'].isin(section_statuses)), ['Status Origin', imm]]
        
        for section_group_id, section_group in enumerate(section_groups):
            
            _, pval = mannwhitneyu(
                df_section.loc[df_section["Status Origin"] == section_group[0], imm].values,
                df_section.loc[df_section["Status Origin"] == section_group[1], imm].values,
                alternative="two-sided",
            )
            bias_0 = np.mean(df_section.loc[df_section['Status Origin'] == section_group[0], imm])
            bias_1 = np.mean(df_section.loc[df_section['Status Origin'] == section_group[1], imm])
            
            imms_tests.at[imm, f"pval\n{section_id}\n{section_group}"] = pval
            imms_tests.at[imm, f"bias_0\n{section_id}\n{section_group}"] = bias_0
            imms_tests.at[imm, f"bias_1\n{section_id}\n{section_group}"] = bias_1

# FDR correction
pvals_cols = [col for col in imms_tests.columns if 'pval' in col]
for imm in (pbar := tqdm(imms)):
    _, imms_tests.loc[imm, pvals_cols], _, _ = multipletests(imms_tests.loc[imm, pvals_cols], 0.05, method='fdr_bh')

for imm in (pbar := tqdm(imms)):
    pbar.set_description(f"Processing {imm}")

    passed_icd_chpt = {icd_chpt: 0 for icd_chpt in icd_chpts}
    passed_icd_chpt_max = {icd_chpt: 0 for icd_chpt in icd_chpts}
    for icd_chpt in icd_chpts:
        df_chpt = df_groups[df_groups['ICD-11 chapter'] == icd_chpt]
        for section_id, section_row in df_chpt.iterrows():
            section_statuses = ast.literal_eval(section_row['Statuses'])
            section_groups = ast.literal_eval(section_row['Groups'])
            section_directions = ast.literal_eval(section_row['Directions'])
            
            for section_group_id, section_group in enumerate(section_groups):
                passed_icd_chpt_max[icd_chpt] += 1
                
                pval = imms_tests.at[imm, f"pval\n{section_id}\n{section_group}"]
                bias_0 = imms_tests.at[imm, f"bias_0\n{section_id}\n{section_group}"]
                bias_1 = imms_tests.at[imm, f"bias_1\n{section_id}\n{section_group}"]
                
                group_direction = section_directions[section_group_id]
                
                if pval < 0.05:
                    passed_icd_chpt[icd_chpt] += 1
                    
        imms_tests.at[imm, f'Passed\nICD-11\nChapter {icd_chpt}'] = passed_icd_chpt[icd_chpt]
        imms_tests.at[imm, f'Max\nPassed\nICD-11\nChapter {icd_chpt}'] = passed_icd_chpt_max[icd_chpt]
                  
    imms_tests.at[imm, f'Passed\nICD-11\nTotal'] = sum(passed_icd_chpt.values())
    imms_tests.at[imm, f'Max\nPassed\nICD-11\nTotal'] = sum(passed_icd_chpt_max.values())

# Plot Figure 6b
imms_tests[f"Passed\nICD-11\nTotal"] /= imms_tests.at['CXCL9', f'Max\nPassed\nICD-11\nTotal']
for col in icd_cols:
    imms_tests[col] /= imms_tests.at['CXCL9', f'Max\n{col}']
imms_tests.index.name = 'Inflammatory marker'

col_names_common = [f"Passed\nICD-11\nTotal"]
col_defs_common = [
    ColumnDefinition(
        name="Inflammatory marker",
        title="Inflammatory\nmarker",
        textprops={"ha": "right", "weight": "bold"},
        width=1.5,
    ),
    ColumnDefinition(
        name=f"Passed\nICD-11\nTotal",
        title="Passed\nICD-11 tests",
        width=1.5,
        border="left",
        textprops={"ha": "center"},
        plot_fn=bar,
        plot_kw={
            "cmap": colormap_total,
            "plot_bg_bar": True,
            "annotate": True,
            "height": 0.95,
            "linewidth": 0.5,
            "formatter": form_bar(imms_tests.at['CXCL9', f'Max\nPassed\nICD-11\nTotal']),
        },
    ),
]

icd_chpt_col_defs = copy.deepcopy(col_defs_common)
icd_chpt_col_names = copy.deepcopy(col_names_common)
for icd_chpt in icd_chpts:
    if icd_chpt == 1:
        border = 'left'
    else:
        border = None
    max_passed = imms_tests.at['CXCL9', f'Max\nPassed\nICD-11\nChapter {icd_chpt}']
    icd_chpt_col_names.append(f'Passed\nICD-11\nChapter {icd_chpt}')
    col_def = ColumnDefinition(
        name=f'Passed\nICD-11\nChapter {icd_chpt}',
        title=f'Chapter {icd_chpt}\n{icd_chpts_names_dict[icd_chpt]}',
        width=1.0,
        plot_fn=bar,
        border=border,
        textprops={"ha": "center"},
        plot_kw={
            "cmap": colormaps_icd_chpts[icd_chpt],
            "plot_bg_bar": True,
            "annotate": True,
            "height": 0.95,
            "lw": 0.5,
            "formatter": form_bar(max_passed),
        },
    )
    icd_chpt_col_defs.append(col_def)

fig, ax = plt.subplots(figsize=(22, 14))
table = Table(
    imms_tests[icd_chpt_col_names],
    column_definitions=icd_chpt_col_defs,
    row_dividers=True,
    footer_divider=False,
    odd_row_color="#ffffff", 
    even_row_color="#f0f0f0",
    ax=ax,
    row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 1))},
    col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
    column_border_kw={"linewidth": 1, "linestyle": "-"},
).autoset_fontcolors(colnames=icd_chpt_col_names) 
fig.savefig(f"{path_plots}/figure6b.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_plots}/figure6b.pdf", bbox_inches='tight')
plt.close(fig)